In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

inp = Path('/kaggle/input')
print('input exists', inp.exists())
if inp.exists():
    for p in sorted(inp.rglob('*')):
        if p.is_file() and p.suffix.lower() in {'.csv', '.json'} or p.name == 'sample_submission.csv':
            print('FILE', p)
        elif p.is_dir() and len(p.parts) <= 4:
            print('DIR', p)

def discover_root() -> Path:
    candidates = []
    if inp.exists():
        for child in sorted(inp.iterdir()):
            candidates.append(child)
            for sub in child.iterdir() if child.is_dir() else []:
                if sub.is_dir():
                    candidates.append(sub)
    for c in candidates:
        if (c / 'sample_submission.csv').exists():
            return c
    # last resort: any sample_submission.csv under input
    hits = list(inp.rglob('sample_submission.csv')) if inp.exists() else []
    if hits:
        return hits[0].parent
    raise FileNotFoundError(f'Competition data not found; candidates={candidates}')

ROOT = discover_root()
print('ROOT', ROOT)
sample = pd.read_csv(ROOT / 'sample_submission.csv')
train_path = ROOT / 'train.csv'
train = pd.read_csv(train_path) if train_path.exists() else None
study_col = 'StudyInstanceUID'
targets = [c for c in sample.columns if c != study_col]
print('n_test', len(sample), 'targets', len(targets))

prev = {}
for t in targets:
    if train is not None and t in train.columns and train[t].notna().any():
        prev[t] = float(train[t].mean())
    else:
        prev[t] = 0.5
print({k: round(v, 4) for k, v in prev.items()})

rng = np.random.default_rng(42)
out = sample[[study_col]].copy()
for t in targets:
    out[t] = np.clip(prev[t] + rng.normal(0.0, 0.01, size=len(sample)), 1e-6, 1 - 1e-6)
out = out[sample.columns]

assert list(out[study_col].astype(str)) == list(sample[study_col].astype(str))
assert not out[targets].isna().any().any()

out_path = Path('/kaggle/working/submission.csv')
out.to_csv(out_path, index=False)
print('Wrote', out_path, out.shape)
print(out.head())